# 00_bootstrap_sec

Fetches all SEC Form 4 insider trading filings for every ticker in
`precursor.bronze.universe` between 2020-01-01 and today, writing to
`precursor.bronze.sec_raw`.

The EDGAR submissions endpoint returns ALL filings for a company in a single
request — no pagination needed. We filter client-side for Form 4 only and
the 2020+ date range. This means only ~614 total HTTP requests, completing
in under 2 minutes.

**Run this notebook ONCE manually.** After bootstrap, `01_ingestion.py`
handles daily updates.

In [0]:
%pip install requests "numpy<2.0" --no-deps

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import pandas as pd
import time
import logging
from datetime import datetime
from typing import Optional

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, LongType,
    TimestampType, BooleanType,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("precursor.sec_bootstrap")

START_DATE = "2020-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

# SEC limit is 10 req/sec; we stay at ~9 req/sec to be safe.
SLEEP_BETWEEN_REQUESTS = 0.11

# SEC EDGAR requires a User-Agent in the format "Name contact@email.com".
# Without it, all requests return 403.
SEC_HEADERS = {
    "User-Agent":       "Precursor Research precursor@gmail.com",
    "Accept-Encoding":  "gzip, deflate",
    "Host":             "data.sec.gov",
}

SEC_TICKER_HEADERS = {
    "User-Agent":       "Precursor Research precursor@gmail.com",
    "Accept-Encoding":  "gzip, deflate",
    "Host":             "www.sec.gov",
}

_SEC_RAW_SCHEMA = StructType([
    StructField("ticker",           StringType(),    nullable=False),
    StructField("cik",              StringType(),    nullable=False),
    StructField("accession_number", StringType(),    nullable=False),
    StructField("transaction_date", DateType(),      nullable=True),
    StructField("filing_date",      DateType(),      nullable=True),
    StructField("form_type",        StringType(),    nullable=False),
    StructField("days_to_file",     LongType(),      nullable=True),
    StructField("is_late_filing",   BooleanType(),   nullable=True),
    StructField("ingested_at",      TimestampType(), nullable=False),
])

logger.info("START_DATE=%s  END_DATE=%s", START_DATE, END_DATE)

INFO:precursor.sec_bootstrap:START_DATE=2020-01-01  END_DATE=2026-05-01


## Cell 3 — Load ticker to CIK mapping

In [0]:
def fetch_cik_mapping() -> dict[str, str]:
    """Fetch the SEC EDGAR ticker → CIK mapping in a single request.

    CIKs are zero-padded to 10 digits as required by the submissions endpoint.
    SEC stores some tickers with "-" instead of "/" (e.g. "BRK-B"), so we
    normalise them to "/" to match our universe table.

    Returns:
        Dict mapping ticker symbol → zero-padded CIK string.
    """
    url  = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(url, headers=SEC_TICKER_HEADERS, timeout=30)
    resp.raise_for_status()

    data    = resp.json()
    mapping = {}
    for entry in data.values():
        ticker = entry["ticker"].replace("-", "/")
        cik    = str(entry["cik_str"]).zfill(10)
        mapping[ticker] = cik

    logger.info("CIK mapping fetched: %d companies.", len(mapping))
    time.sleep(SLEEP_BETWEEN_REQUESTS)
    return mapping


def load_universe_tickers() -> list[str]:
    """Load all distinct tickers from precursor.bronze.universe.

    Returns:
        Sorted list of ticker symbols that were ever in the index.
    """
    df      = spark.sql("""
        SELECT DISTINCT ticker
        FROM precursor.bronze.universe
        WHERE in_index = TRUE
        ORDER BY ticker
    """).toPandas()
    tickers = df["ticker"].tolist()
    logger.info("Universe tickers loaded: %d total.", len(tickers))
    return tickers


def match_tickers_to_ciks(
    universe_tickers: list[str],
    cik_mapping: dict[str, str],
) -> tuple[dict[str, str], list[str]]:
    """Match universe tickers to their SEC CIKs.

    Args:
        universe_tickers: List of ticker symbols from the universe table.
        cik_mapping:      Dict of ticker → CIK from EDGAR.

    Returns:
        Tuple of (matched dict[ticker → CIK], unmatched list[ticker]).
    """
    matched:   dict[str, str] = {}
    unmatched: list[str]      = []

    for ticker in universe_tickers:
        if ticker in cik_mapping:
            matched[ticker] = cik_mapping[ticker]
        else:
            unmatched.append(ticker)

    logger.info("Matched: %d tickers.  Unmatched: %d tickers.", len(matched), len(unmatched))
    if unmatched:
        logger.warning("Unmatched tickers (no CIK found): %s", unmatched)

    return matched, unmatched

## Cell 4 — Fetch Form 4 filings for one ticker

In [0]:
def fetch_insider_filings(
    ticker: str,
    cik: str,
    start_date: str,
    end_date: str,
) -> Optional[pd.DataFrame]:
    """Fetch all Form 4 insider trading filings for one company.

    Makes a single request to the EDGAR submissions endpoint which returns
    ALL filings for the company in one response. We filter client-side for
    Form 4 only and the 2020+ date window.

    Transaction date vs filing date — look-ahead bias prevention:
        Form 4 must be filed within 2 business days of the transaction.
        We use reportDate (the transaction date) as the feature date, NOT
        filingDate. A CEO buying stock on Monday and filing on Wednesday
        should be attributed to Monday — using Wednesday would mean we
        "know" about the trade before it was actually public, introducing
        look-ahead bias into any model trained on this data.

    Older filings in "files" array:
        EDGAR splits filings into "recent" (latest ~1000) and older batches
        referenced in a "files" array. For our 2020+ date range the "recent"
        array is always sufficient — companies rarely have >1000 Form 4s
        since 2020. Fetching the "files" batches would require additional
        requests and is deferred to a future version.

    Args:
        ticker:     Ticker symbol.
        cik:        Zero-padded 10-digit CIK string.
        start_date: Filter start date (YYYY-MM-DD).
        end_date:   Filter end date (YYYY-MM-DD).

    Returns:
        DataFrame with filing metadata, or None if no Form 4s found or
        the request failed.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    try:
        resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
        resp.raise_for_status()
    except Exception as exc:
        logger.warning("%s: request failed — %s", ticker, exc)
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        return None
    finally:
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    try:
        data    = resp.json()
        recent  = data.get("filings", {}).get("recent", {})

        forms        = recent.get("form",            [])
        filing_dates = recent.get("filingDate",      [])
        report_dates = recent.get("reportDate",      [])
        accessions   = recent.get("accessionNumber", [])

        if not forms:
            logger.info("%s: no filings found in submissions response.", ticker)
            return None

        records = []
        for form, filing_date, report_date, accession in zip(
            forms, filing_dates, report_dates, accessions
        ):
            if form != "4":
                continue
            if filing_date < start_date or filing_date > end_date:
                continue
            records.append({
                "ticker":           ticker,
                "cik":              cik,
                "accession_number": accession,
                "transaction_date": report_date  if report_date  else None,
                "filing_date":      filing_date  if filing_date  else None,
                "form_type":        "4",
                "ingested_at":      datetime.utcnow(),
            })

        if not records:
            logger.info("%s: no Form 4 filings found in date range.", ticker)
            return None

        df = pd.DataFrame(records)
        df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce").dt.date
        df["filing_date"]      = pd.to_datetime(df["filing_date"],      errors="coerce").dt.date

        return df

    except Exception as exc:
        logger.warning("%s: failed to parse response — %s", ticker, exc)
        return None

## Cell 5 — Build filing features

In [0]:
# TODO V2: parse individual Form 4 XML documents for transaction type (buy/sell),
# number of shares, and price per share. Each company requires 20-100 additional
# requests to fetch individual XML documents. For V1 we use filing frequency as
# the signal — insider filing count over 7/30 day windows is itself a valid
# predictor of informed trading activity without needing transaction-level detail.

def build_filing_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived quality and feature columns to the raw filings DataFrame.

    Args:
        df: Raw filings DataFrame from fetch_insider_filings.

    Returns:
        Enriched DataFrame with days_to_file and is_late_filing columns.
    """
    df = df.copy()

    filing_dt      = pd.to_datetime(df["filing_date"],      errors="coerce")
    transaction_dt = pd.to_datetime(df["transaction_date"], errors="coerce")

    df["days_to_file"]   = (filing_dt - transaction_dt).dt.days
    df["is_late_filing"] = df["days_to_file"] > 2

    return df

## Cell 6 — Process all tickers and write to Delta

In [0]:
def process_all_tickers(
    ticker_cik_map: dict[str, str],
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """Fetch Form 4 filings for all matched tickers and combine into one DataFrame.

    Args:
        ticker_cik_map: Dict of ticker → CIK for all matched tickers.
        start_date:     Filter start date (YYYY-MM-DD).
        end_date:       Filter end date (YYYY-MM-DD).

    Returns:
        Combined DataFrame of all filings with feature columns added.
    """
    all_frames: list[pd.DataFrame] = []
    total = len(ticker_cik_map)

    for i, (ticker, cik) in enumerate(ticker_cik_map.items(), start=1):
        df = fetch_insider_filings(ticker, cik, start_date, end_date)
        if df is not None:
            all_frames.append(df)

        if i % 50 == 0:
            logger.info("Progress: %d/%d tickers processed.", i, total)

    if not all_frames:
        raise RuntimeError("No Form 4 filings returned for any ticker.")

    combined = pd.concat(all_frames, ignore_index=True)
    combined = build_filing_features(combined)

    logger.info(
        "All tickers processed: %d filings across %d tickers.",
        len(combined), combined["ticker"].nunique(),
    )
    return combined


def write_sec_to_delta(df: pd.DataFrame) -> None:
    """Convert combined filings DataFrame to Spark and write to sec_raw.

    Args:
        df: Combined filings DataFrame from process_all_tickers.
    """
    df = df.copy()
    df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
    df["filing_date"]      = pd.to_datetime(df["filing_date"],      errors="coerce")
    df["ingested_at"]      = pd.to_datetime(df["ingested_at"])
    df["days_to_file"]     = df["days_to_file"].astype("Int64").apply(
        lambda x: int(x) if pd.notna(x) else None
    )
    df["is_late_filing"]   = df["is_late_filing"].astype("boolean").apply(
        lambda x: bool(x) if pd.notna(x) else None
    )

    keep     = ["ticker", "cik", "accession_number", "transaction_date",
                "filing_date", "form_type", "days_to_file", "is_late_filing", "ingested_at"]
    spark_df = spark.createDataFrame(df[keep], schema=_SEC_RAW_SCHEMA)

    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("precursor.bronze.sec_raw")
    )

    written = spark.table("precursor.bronze.sec_raw").count()
    logger.info("precursor.bronze.sec_raw written: %d rows.", written)

## Cell 7 — Main execution

In [0]:
_bootstrap_start = datetime.now()
logger.info("=== precursor.00_bootstrap_sec START at %s ===", _bootstrap_start.isoformat())

try:
    _cik_mapping      = fetch_cik_mapping()
    _universe_tickers = load_universe_tickers()
    _ticker_cik_map, _unmatched = match_tickers_to_ciks(_universe_tickers, _cik_mapping)

    _n = len(_ticker_cik_map)
    logger.info("Processing %d matched tickers.", _n)
    logger.info("Estimated time: %.1f minutes.", _n * SLEEP_BETWEEN_REQUESTS / 60)

    _combined = process_all_tickers(_ticker_cik_map, START_DATE, END_DATE)
    write_sec_to_delta(_combined)

    _elapsed = (datetime.now() - _bootstrap_start).total_seconds() / 60
    logger.info("=== precursor.00_bootstrap_sec END — %.2f min total ===", _elapsed)

    _tickers_with_data    = _combined["ticker"].nunique()
    _tickers_without_data = _n - _tickers_with_data
    _date_min             = _combined["transaction_date"].min()
    _date_max             = _combined["transaction_date"].max()

    print("=" * 60)
    print("  PRECURSOR — SEC BOOTSTRAP SUMMARY")
    print("=" * 60)
    print(f"  Total tickers attempted    : {_n}")
    print(f"  Tickers with Form 4 data   : {_tickers_with_data}")
    print(f"  Tickers with no filings    : {_tickers_without_data}")
    print(f"  Unmatched tickers (no CIK) : {len(_unmatched)}")
    print(f"  Total filings written      : {len(_combined):,}")
    print(f"  Filing date range          : {_date_min} → {_date_max}")
    print(f"  Elapsed time               : {_elapsed:.2f} min")
    print("=" * 60)

except Exception as exc:
    logger.error("Bootstrap failed: %s", exc, exc_info=True)

INFO:precursor.sec_bootstrap:=== precursor.00_bootstrap_sec START at 2026-05-01T18:41:23.429117 ===
INFO:precursor.sec_bootstrap:CIK mapping fetched: 10348 companies.
INFO:precursor.sec_bootstrap:Universe tickers loaded: 613 total.
INFO:precursor.sec_bootstrap:Matched: 562 tickers.  Unmatched: 51 tickers.
INFO:precursor.sec_bootstrap:Processing 562 matched tickers.
INFO:precursor.sec_bootstrap:Estimated time: 1.0 minutes.
INFO:precursor.sec_bootstrap:Progress: 50/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 100/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 150/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 200/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 250/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 300/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 350/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 400/562 tickers processed.
INFO:precursor.sec_bootstrap:Progress: 450/562 

  PRECURSOR — SEC BOOTSTRAP SUMMARY
  Total tickers attempted    : 562
  Tickers with Form 4 data   : 562
  Tickers with no filings    : 0
  Unmatched tickers (no CIK) : 51
  Total filings written      : 227,307
  Filing date range          : 2001-01-01 → 2026-05-01
  Elapsed time               : 3.21 min


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read current table
df = spark.read.table("precursor.bronze.sec_raw")

print(f"Rows before fix: {df.count()}")

# Fix 1 — apply date filter
df = df.filter(F.col("transaction_date") >= "2020-01-01")
print(f"Rows after date filter: {df.count()}")

# Fix 2 — deduplicate on accession_number
# keep first occurrence per accession_number
window = Window.partitionBy("accession_number") \
               .orderBy("ticker")

df = df.withColumn("row_num", F.row_number().over(window)) \
       .filter(F.col("row_num") == 1) \
       .drop("row_num")

print(f"Rows after dedup: {df.count()}")

# Overwrite the table with clean data
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable("precursor.bronze.sec_raw")

print("✅ sec_raw fixed and overwritten")

Rows before fix: 227307
Rows after date filter: 226608
Rows after dedup: 225346
✅ sec_raw fixed and overwritten


## Cell 8 — Validation

In [0]:
logger.info("=== Running post-bootstrap validation ===")

_checks: list[tuple[str, bool, str]] = []

def _check(name: str, passed: bool, detail: str = "") -> None:
    status = "PASS" if passed else "FAIL"
    msg    = f"[{status}] {name}" + (f" — {detail}" if detail else "")
    logger.info(msg)
    _checks.append((name, passed, detail))


try:
    from datetime import timedelta

    # 1. Row count > 0
    row_count = spark.table("precursor.bronze.sec_raw").count()
    _check("Row count > 0", row_count > 0, f"{row_count:,} rows")

    # 2. At least 400 distinct tickers
    distinct_tickers = spark.sql(
        "SELECT COUNT(DISTINCT ticker) AS n FROM precursor.bronze.sec_raw"
    ).collect()[0]["n"]
    _check("At least 400 distinct tickers", distinct_tickers >= 400, f"{distinct_tickers} tickers")

    # 3. transaction_date range spans 2020 to today
    min_year, max_year = spark.sql("""
        SELECT MIN(YEAR(transaction_date)) AS mn, MAX(YEAR(transaction_date)) AS mx
        FROM precursor.bronze.sec_raw
    """).collect()[0]
    _check(
        "Date range spans 2020 to current year",
        min_year is not None and min_year <= 2020 and max_year >= datetime.today().year,
        f"years {min_year}–{max_year}",
    )

    # 4. No null transaction_dates
    null_dates = spark.sql(
        "SELECT COUNT(*) AS n FROM precursor.bronze.sec_raw WHERE transaction_date IS NULL"
    ).collect()[0]["n"]
    _check("No null transaction_dates", null_dates == 0, f"{null_dates} nulls")

    # 5. All form_type == "4"
    bad_forms = spark.sql(
        "SELECT COUNT(*) AS n FROM precursor.bronze.sec_raw WHERE form_type != '4'"
    ).collect()[0]["n"]
    _check("All form_type == '4'", bad_forms == 0, f"{bad_forms} non-4 rows")

    # 6. days_to_file between 0 and 10 for 95%+ of rows
    total_rows = spark.sql("SELECT COUNT(*) AS n FROM precursor.bronze.sec_raw").collect()[0]["n"]
    in_range   = spark.sql("""
        SELECT COUNT(*) AS n FROM precursor.bronze.sec_raw
        WHERE days_to_file BETWEEN 0 AND 10
    """).collect()[0]["n"]
    pct = in_range / total_rows * 100 if total_rows > 0 else 0
    _check(
        "95%+ rows have days_to_file in [0, 10]",
        pct >= 95,
        f"{pct:.1f}% in range",
    )

    # 7. No duplicate accession numbers
    total_acc    = spark.sql("SELECT COUNT(*) AS n FROM precursor.bronze.sec_raw").collect()[0]["n"]
    distinct_acc = spark.sql(
        "SELECT COUNT(DISTINCT accession_number) AS n FROM precursor.bronze.sec_raw"
    ).collect()[0]["n"]
    _check("No duplicate accession_numbers", total_acc == distinct_acc,
           f"{total_acc - distinct_acc} duplicates")

    # 8. Latest filing within last 30 days
    latest = spark.sql(
        "SELECT MAX(filing_date) AS d FROM precursor.bronze.sec_raw"
    ).collect()[0]["d"]
    thirty_days_ago = (datetime.today() - timedelta(days=30)).date()
    _check(
        "Latest filing within last 30 days",
        latest is not None and latest >= thirty_days_ago,
        f"latest = {latest}",
    )

except Exception as exc:
    logger.error("Validation query failed: %s", exc, exc_info=True)
    _checks.append(("Validation queries executed without error", False, str(exc)))

_passed_n = sum(1 for _, p, _ in _checks if p)
_failed_n = len(_checks) - _passed_n

print("=" * 60)
print("  VALIDATION RESULTS")
print("=" * 60)
for name, passed, detail in _checks:
    status = "PASS" if passed else "FAIL"
    line   = f"  [{status}] {name}"
    if detail:
        line += f"\n         {detail}"
    print(line)
print("-" * 60)
print(f"  {_passed_n} passed  /  {_failed_n} failed")
if _failed_n > 0:
    print("  WARNING: validation failures detected — review logs above.")
print("=" * 60)

INFO:precursor.sec_bootstrap:=== Running post-bootstrap validation ===
INFO:precursor.sec_bootstrap:[PASS] Row count > 0 — 227,307 rows
INFO:precursor.sec_bootstrap:[PASS] At least 400 distinct tickers — 562 tickers
INFO:precursor.sec_bootstrap:[PASS] Date range spans 2020 to current year — years 2001–2026
INFO:precursor.sec_bootstrap:[PASS] No null transaction_dates — 0 nulls
INFO:precursor.sec_bootstrap:[PASS] All form_type == '4' — 0 non-4 rows
INFO:precursor.sec_bootstrap:[PASS] 95%+ rows have days_to_file in [0, 10] — 98.8% in range
INFO:precursor.sec_bootstrap:[FAIL] No duplicate accession_numbers — 1264 duplicates
INFO:precursor.sec_bootstrap:[PASS] Latest filing within last 30 days — latest = 2026-05-01


  VALIDATION RESULTS
  [PASS] Row count > 0
         227,307 rows
  [PASS] At least 400 distinct tickers
         562 tickers
  [PASS] Date range spans 2020 to current year
         years 2001–2026
  [PASS] No null transaction_dates
         0 nulls
  [PASS] All form_type == '4'
         0 non-4 rows
  [PASS] 95%+ rows have days_to_file in [0, 10]
         98.8% in range
  [FAIL] No duplicate accession_numbers
         1264 duplicates
  [PASS] Latest filing within last 30 days
         latest = 2026-05-01
------------------------------------------------------------
  7 passed  /  1 failed
